<a href="https://colab.research.google.com/github/mugalan/intrinsic-rigid-body-control-estimation/blob/main/intrinsic-DEKF/RigidBodyIntinsicEKF_DHSM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports and Setup

In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
from scipy.integrate import odeint
import math
from numpy import linalg
import sympy
from sympy import symbols
from sympy import *

import plotly.graph_objects as go
import plotly.express as px
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
from IPython.display import display, Math, Latex

In [ ]:
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"
import sims
mr = sims.RigidBodySim()

# The Kalman Filter on $\mathbb{R}^n$

Consider the linear Gaussian process:
\begin{align*}
x_k &= A_{k-1}\,x_{k-1} + G_{k-1}\,w_{k-1}, \\
y_k &= H_k\,x_k + z_k,
\end{align*}
where $w_k \sim \mathscr{N}(0,\Sigma_p)$ and $z_k \sim \mathscr{N}(0,\Sigma_m)$ are mutually independent white noise sequences, also independent of the initial state $x_0 \sim \mathscr{N}(m_0, P_0)$.
Here $y_k$ denotes the random variable representing the measurement at time step $k$, while $y_k^{\mathrm{obs}}$ denotes its observed numerical realization.

---

**Prediction Step (Time Update):**

Define the filter model
\begin{align*}
x^{-}_k &= A_{k-1}\,x^{+}_{k-1} + G_{k-1}\,w_{k-1}, \\
y^{-}_k &= H_k\,x^{-}_k + z_k,
\end{align*}
where
$$x^{+}_{k-1} \sim \mathscr{N}(m_{k-1}, P_{k-1}),$$


Because the filter equation is linear and the process noise is Gaussian, the predicted state $x^{-}_k$ is also Gaussian. Taking the expectation of the state equation yields the predicted mean:


$$m_k^- \triangleq \mathbb{E}[x^{-}_k] = A_{k-1}\mathbb{E}[x^{+}_{k-1}] + G_{k-1}\mathbb{E}[w_{k-1}] = A_{k-1}m_{k-1},$$


since $\mathbb{E}[w_{k-1}] = 0$.

The predicted covariance $P_k^-$ is computed by applying the variance operator to the state equation:
\begin{align*}
P_k^- &\triangleq \text{Var}(x^{-}_k) \\
&= A_{k-1}\text{Var}(x^{+}_{k-1})A_{k-1}^T + G_{k-1}\text{Var}(w_{k-1})G_{k-1}^T \\
&= A_{k-1}P_{k-1}A_{k-1}^T + G_{k-1}\Sigma_p G_{k-1}^T.
\end{align*}
Thus, prior to incorporating the new measurement, our belief of the state is characterized by the prior distribution:


$$x_k^- \sim \mathscr{N}(m_k^-, P_k^-).$$

---

**Measurement Prediction:**

Prior to its physical realization, the upcoming measurement at time step $k$ is treated as the random variable $y_k$. Based on our prior state belief, its expected value is:


$$\mathbb{E}[y^{-}_k] = H_k \mathbb{E}[x_k^-] + \mathbb{E}[z_k] = H_k m_k^-,$$


and its variance is:


$$\text{Var}(y^{-}_k) = H_k \text{Var}(x_k^-) H_k^T + \text{Var}(z_k) = H_k P_k^- H_k^T + \Sigma_m.$$

---

**Joint Prior Distribution:**

The cross-covariance between the predicted state $x_k^-$ and the anticipated measurement random variable $y_k$ is evaluated as:


$$\text{Cov}(x_k^-, y^{-}_k) = \text{Cov}(x_k^-, H_k x_k^- + z_k) = P_k^- H_k^T.$$

Therefore, the joint distribution of the predicted state and the upcoming measurement is a block-structured multivariate Gaussian:
$$\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathscr{N}\left(
\begin{bmatrix} m_k^- \\ H_k m_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^- H_k^T \\ H_k P_k^- & H_k P_k^- H_k^T + \Sigma_m \end{bmatrix}
\right).$$

---

**Measurement Update (Correction Step):**

At time step $k$, a physical measurement is observed, causing the random variable to take a specific numerical realization: $y_k = y^{\mathrm{obs}}_{k}$.

Applying the standard conditioning properties of multivariate normal distributions, we update our prior belief of the state by slicing the joint distribution at the realization $y^{\mathrm{obs}}_{k}$. This yields the posterior state distribution:


$$x^{+}_k\triangleq (x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}) \sim \mathscr{N}(m_k, P_k),$$


where the updated mean $m_k$ and updated covariance $P_k$ are given by:
\begin{align*}
K_k &\triangleq P_k^- H_k^T (H_k P_k^- H_k^T + \Sigma_m)^{-1}, \\
m_k &= m_k^- + K_k (y^{\mathrm{obs}}_{k} - H_k m_k^-), \\
P_k &= (I - K_k H_k) P_k^-.
\end{align*}

Here, $K_k$ is the Kalman Gain, and
$$ m_k=\mathbb{E}[x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}]$$ acts as the updated state estimate, and
$$P_k=\text{Var}(x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k})$$ is the new posterior error covariance matrix used to seed the next recursive time-step.

Note that $x^{+}_k$ denotes an abstract random variable distributed according to the conditional law of $x^{-}_k$ given the observed value $y^{\mathrm{obs}}_{k}$.

You may refer to this note on [multivariate Gaussians](https://github.com/mugalan/introduction-to-statistical-learning/blob/main/Multivariate_Gaussian_Distributions.ipynb) for the exact details of extracting the mean and the covariance of the conditional distribution.

---

**Error Dynamics**

For clarity, define the actual estimation error by
$$
\varepsilon_k \triangleq x_k - m_k,
\qquad
\varepsilon_k^- \triangleq x_k - m_k^- .
$$

From the measurement update,
$$
m_k = m_k^- + K_k\left(y_k^{\mathrm{obs}} - H_km_k^-\right),
$$
and using
$$
y_k^{\mathrm{obs}} = H_kx_k + z_k,
$$
we obtain
$$
\begin{aligned}
\varepsilon_k
&= x_k - m_k \\
&= x_k - m_k^- - K_k\left(H_kx_k + z_k - H_km_k^-\right) \\
&= \left(I-K_kH_k\right)(x_k-m_k^-) - K_kz_k \\
&= \left(I-K_kH_k\right)\varepsilon_k^- - K_kz_k .
\end{aligned}
$$

Moreover, from the prediction step,
$$
x_k = A_{k-1}x_{k-1} + G_{k-1}w_{k-1},
\qquad
m_k^- = A_{k-1}m_{k-1},
$$
we have
$$
\varepsilon_k^-
=
A_{k-1}\varepsilon_{k-1}
+
G_{k-1}w_{k-1}.
$$

Therefore,
$$
\boxed{
\varepsilon_k
=
\left(I-K_kH_k\right)A_{k-1}\varepsilon_{k-1}
+
\left(I-K_kH_k\right)G_{k-1}w_{k-1}
-
K_kz_k .
}
$$

## 1-D Example

Consider the scalar linear-Gaussian filter model:
\begin{aligned}
x^-_k &= a\,x^+_{k-1} + w_{k-1},\qquad w_{k-1}\sim\mathscr N(0,q),\\
y^-_k &= h\,x^-_k + z_k,\qquad\;\;\;\; z_k\sim\mathscr N(0,r),
\end{aligned}
with prior $x_0\sim \mathscr{N}(m_0,P_0)$. Define $Y_{k}=\{y^{\mathrm{obs}}_1,\dots,y^{\mathrm{obs}}_{k}\}$.

Prediction:

\begin{aligned}
m_k^- &= a\,m_{k-1},\\
P_k^- &= a^2 P_{k-1} + q.
\end{aligned}

Innovation (measurement) model and variance:
\begin{aligned}
v_k &\triangleq y^{\mathrm{obs}}_k - h\,m_k^-,\\
S_k &= h^2 P_k^- + r.
\end{aligned}

Kalman gain:
\begin{align*}
K_k = \dfrac{P_k^-\,h}{S_k}.
\end{align*}

Update:
\begin{aligned}
m_k &= m_k^- + K_k\,v_k
= m_k^- + \frac{P_k^- h}{S_k}\bigl(y^{\mathrm{obs}}_k - h\,m_k^-\bigr),\\
P_k &= (1 - K_k h)\,P_k^-
= \Bigl(1 - \frac{P_k^- h^2}{S_k}\Bigr) P_k^- .
\end{aligned}

Predictive measurement distribution (before seeing y_k):
\begin{align*}
p(y^-_k | Y_{k-1})=\mathscr N\bigl(h\,m_k^-,\; h^2 P_k^- + r\bigr).
\end{align*}

Posterior-predictive measurement distribution (after filtering on y_k):
\begin{align*}
p(y^-_k\mid Y_k)=\mathscr N\bigl(h\,m_k,\; h^2 P_k + r\bigr).
\end{align*}



In [ ]:
import numpy as np

def make_cv1d(dt: float = 0.1,
              q: float = 1e-2,
              r: float = 1e-1,
              x0=(0.0, 1.0),
              seed: int | None = None) -> sims.LinearGaussianSystemSyms:
    """
    Build a 1D constant-velocity linear Gaussian system:

        x_k = A x_{k-1} + w_{k-1},      w ~ N(0, Q)
        y_k = H x_k       + z_k,        z ~ N(0, R)

    State: x = [position, velocity]^T  (n=2)
    Measurement: y = position (scalar, p=1)

    Parameters
    ----------
    dt : float
        Sampling period Δt.
    q : float
        Continuous white-acceleration noise intensity (process noise scale).
        Discrete-time Q = q * [[dt^3/3, dt^2/2],
                               [dt^2/2, dt     ]].
    r : float
        Measurement noise std. R = [[r^2]] (scalar variance).
    x0 : tuple[float, float]
        Initial state (position, velocity).
    seed : int | None
        Seed for reproducible randomness.

    Returns
    -------
    LinearGaussianSystemSyms
        System with A, H, Sigma_p (Q), Sigma_m (R), and initial state x0.
    """
    A = np.array([[1.0, dt],
                  [0.0, 1.0]], dtype=float)

    # Measure position only (scalar)
    H = np.array([[1.0, 0.0]], dtype=float)  # shape (1,2)

    # Discrete CV process noise covariance (from white-acceleration model)
    Q = q * np.array([[dt**3/3.0, dt**2/2.0],
                      [dt**2/2.0, dt      ]], dtype=float)

    # Measurement noise covariance (scalar)
    R = np.array([[r**2]], dtype=float)

    x0 = np.asarray(x0, dtype=float)
    if x0.shape != (2,):
        raise ValueError(f"`x0` must be shape (2,), got {x0.shape}.")

    rng = np.random.default_rng(seed)
    return sims.LinearGaussianSystemSyms(A=A, H=H, Sigma_p=Q, Sigma_m=R, x0=x0, rng=rng)

# Must have p=1 in your system (scalar measurement). n can be >1.
sys = make_cv1d(dt=0.1, q=5e-2, r=1.0, x0=(0.0, 0.5), seed=7)  # p=1
# Run with simulated Y (T provided)
sys.animate_measurement_gaussians_scalar(T=80, m0=np.zeros(sys.n), P0=np.eye(sys.n)*100,
                                         frame_ms=120, save_html_path=None, show=True)

# Or if you've already collected Y (shape (T,) or (T,1)):
# _, Y = sys.simulate(T=100)
# sys.animate_measurement_gaussians_scalar(Y=Y, m0=np.zeros(sys.n), P0=np.eye(sys.n)*100,
#                                          save_html_path="kf_scalar_y_gaussians.html", auto_play=False)

## Simulation 2D-example

We consider a two-dimensional constant-velocity dynamical system. The hidden state at time step $k$ is

$$
x_k =
\begin{bmatrix}
p_x(k)\\
p_y(k)\\
v_x(k)\\
v_y(k)
\end{bmatrix},
$$

where $(p_x(k),p_y(k))$ denote the position components and $(v_x(k),v_y(k))$ denote the velocity components.

The measurement consists only of the two position components:

$$
y_k =
\begin{bmatrix}
p_x^{\mathrm{meas}}(k)\\
p_y^{\mathrm{meas}}(k)
\end{bmatrix}.
$$

The linear Gaussian state-space model is

$$
x_k = A x_{k-1} + G w_{k-1},
$$

$$
y_k = Hx_k + z_k,
$$

where

$$
w_{k-1} \sim \mathscr{N}(0,\Sigma_p),
\qquad
z_k \sim \mathscr{N}(0,\Sigma_m).
$$

The process noise sequence $w_k$, measurement noise sequence $z_k$, and the initial state are assumed mutually independent.

---


Assuming a sampling interval $\Delta t$, the constant-velocity kinematic equations are

$$
p_x(k) = p_x(k-1) + \Delta t\,v_x(k-1),
$$

$$
p_y(k) = p_y(k-1) + \Delta t\,v_y(k-1),
$$

$$
v_x(k) = v_x(k-1),
$$

$$
v_y(k) = v_y(k-1).
$$

Therefore, in matrix form,

$$
x_k =
\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}
x_{k-1}
+
G w_{k-1}.
$$

Thus,

$$
A =
\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}.
$$

---

Since the measurement contains only the position components, we have

$$
y_k =
\begin{bmatrix}
p_x(k)\\
p_y(k)
\end{bmatrix}
+
z_k.
$$

Equivalently,

$$
y_k = Hx_k + z_k,
$$

where

$$
H =
\begin{bmatrix}
1 & 0 & 0 & 0\\
0 & 1 & 0 & 0
\end{bmatrix}.
$$

The measurement noise is modeled as

$$
z_k \sim \mathscr{N}(0,\Sigma_m),
$$

with

$$
\Sigma_m = r^2 I_2.
$$

Here, $r$ is the standard deviation of the measurement noise in each position coordinate.

---


A common way to model uncertainty in a constant-velocity system is to assume that the unmodeled acceleration is random. Let

$$
w_{k-1} =
\begin{bmatrix}
a_x(k-1)\\
a_y(k-1)
\end{bmatrix},
$$

where

$$
w_{k-1} \sim \mathscr{N}(0,q^2I_2).
$$

Here, $q$ controls the acceleration-noise intensity.

Over one sampling interval $\Delta t$, the random acceleration affects both position and velocity:

$$
p_x(k)
=
p_x(k-1)
+
\Delta t\,v_x(k-1)
+
\frac{1}{2}\Delta t^2 a_x(k-1),
$$

$$
p_y(k)
=
p_y(k-1)
+
\Delta t\,v_y(k-1)
+
\frac{1}{2}\Delta t^2 a_y(k-1),
$$

$$
v_x(k)
=
v_x(k-1)
+
\Delta t\,a_x(k-1),
$$

$$
v_y(k)
=
v_y(k-1)
+
\Delta t\,a_y(k-1).
$$

Therefore,

$$
G =
\begin{bmatrix}
\frac{1}{2}\Delta t^2 & 0\\
0 & \frac{1}{2}\Delta t^2\\
\Delta t & 0\\
0 & \Delta t
\end{bmatrix},
$$

and

$$
\Sigma_p = q I_2.
$$

The induced state-space process covariance is

$$
Q
=
G\Sigma_pG^T.
$$

Since $\Sigma_p=qI_2$, this becomes

$$
Q
=
qGG^T.
$$

Explicitly,

$$
Q
=
q
\begin{bmatrix}
\frac{1}{4}\Delta t^4 & 0 & \frac{1}{2}\Delta t^3 & 0\\
0 & \frac{1}{4}\Delta t^4 & 0 & \frac{1}{2}\Delta t^3\\
\frac{1}{2}\Delta t^3 & 0 & \Delta t^2 & 0\\
0 & \frac{1}{2}\Delta t^3 & 0 & \Delta t^2
\end{bmatrix}.
$$

---

Thus, the full Gaussian filter is

$$
x^-_k =
Ax^+_{k-1}+Gw_{k-1},
\qquad
w_{k-1}\sim\mathscr{N}(0,q^2I_2),
$$

$$
y^-_k =
Hx^-_k+z_k,
\qquad
z_k\sim\mathscr{N}(0,r^2I_2).
$$

In [ ]:
# 2D position-only measurements (p=2), CV model in x & y
def make_cv2d(
    dt=0.1,
    q=1e-4,
    r=0.5,
    x0=(0, 0, 1, 0.5),
    seed=0,
    *,
    use_G=True,
    noise_model="accel_white",
):
    """
    Build a 2D constant-velocity (CV) linear-Gaussian system.

    States: x = [x, y, vx, vy]^T
      A = [[1, 0, dt, 0 ],
           [0, 1, 0 , dt],
           [0, 0, 1 , 0 ],
           [0, 0, 0 , 1 ]]

    Measurements: y = [x, y]^T  (position only)

    Process-noise options
    ---------------------
    - use_G=True, noise_model='accel_white'  (default):
        Uses an explicit G that maps a 2D white acceleration noise (w ~ N(0, q I_2))
        into the state:
            G = [[dt^2/2,     0   ],
                 [   0  ,  dt^2/2],
                 [  dt ,     0   ],
                 [   0 ,    dt   ]]
        Sigma_p = q * I_2   (in w-space)
        → State-space covariance is G Sigma_p G^T
        (This is the common “white-acceleration” CV model.)

    - use_G=False:
        Legacy behavior (no G). We provide the classic state-space Q directly
        corresponding to (velocity random-walk discretization):
            Q_block = [[dt^3/3, dt^2/2],
                       [dt^2/2, dt     ]] * q
        Q = blockdiag(Q_block, Q_block)
        Sigma_p = Q  (already in state space), G=None

    Notes
    -----
    The two variants imply slightly different discrete-time process covariances.
    Pick the one that matches your physical assumption / reference text.
    """
    A = np.array([
        [1, 0, dt,  0],
        [0, 1,  0, dt],
        [0, 0,  1,  0],
        [0, 0,  0,  1],
    ])
    H = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
    ])
    R = np.eye(2) * (r**2)

    if use_G:
        if noise_model != "accel_white":
            raise ValueError("When use_G=True, supported noise_model is only 'accel_white'.")
        # White acceleration injected into vx, vy
        G = np.array([
            [0.5*dt*dt, 0.0       ],
            [0.0      , 0.5*dt*dt],
            [dt       , 0.0       ],
            [0.0      , dt        ],
        ])
        Sigma_p = np.eye(2) * (q**2)  # w-space covariance (2x2)
        return sims.LinearGaussianSystemSyms(
            A=A, H=H, Sigma_p=Sigma_p, Sigma_m=R,
            x0=np.asarray(x0, float),
            rng=np.random.default_rng(seed),
            G=G
        )
    else:
        # Legacy/state-space Q (velocity random-walk discretization)
        Q_block = np.array([[dt**3/3, dt**2/2],
                            [dt**2/2, dt      ]], dtype=float) * q
        Q = np.block([
            [Q_block,               np.zeros((2, 2))],
            [np.zeros((2, 2)),      Q_block        ],
        ])
        return sims.LinearGaussianSystemSyms(
            A=A, H=H, Sigma_p=Q, Sigma_m=R,
            x0=np.asarray(x0, float),
            rng=np.random.default_rng(seed),
            G=G
        )

In [ ]:
sys = make_cv2d(dt=0.1, q=1e-2, r=0.5, x0=(0,0, 0.5, -0.2), seed=7)
X2, Y2 = sys.simulate(T=300)
sys.plot_y(Y2, nbins=40, component_labels=["pos_x", "pos_y"])

In [ ]:
# Option A: simulate 300 steps internally, starting from broad prior
M, Yhat= sys.filter_with_kf_and_plot(T=300, Y=Y2, m0=np.array([0,0, 0,0]), P0=np.eye(4)*100,
                                      component_labels=["pos_x","pos_y"], show=True)

# Option B: if you already have measurements Y, just pass them:
# X, Y = sys.simulate(T=300)
# M, Yhat = sys.filter_with_kf_and_plot(Y=Y, m0=np.array([0,0, 0,0]), P0=np.eye(4)*100,
#                                       component_labels=["pos_x","pos_y"])

# Discrete-Time Pre-Observers on Lie Groups with Time-Invariant Error Dynamics

We now move from linear systems on $\mathbb R^n$ to kinematic systems evolving on a Lie group $G$.
The purpose of this section is to introduce discrete-time pre-observers whose estimation error evolves autonomously, or equivalently, independently of the true trajectory.
This time-invariant error structure is obtained by choosing the observer and the output error consistently with the left- or right-invariance of the system output.


Let $G$ be an $n$-dimensional Lie group with Lie algebra $G$. Let $(g,\zeta)\in G\times G$ and let $\phi:G\times M\to M$ be a left- or right-invariant action on an $m$-dimensional manifold $M$. We consider the kinematic system
\begin{align*}
\dot g &= g\cdot \zeta,  \\
y &= \phi_g(\gamma),
\end{align*}
where $\zeta(t)\in G$ is a known input and $\gamma\in M$ is a known constant. A standard discrete-time approximation is
\begin{align*}
g_k &= g_{k-1}\exp(\Delta T\zeta_{k-1}),
\end{align*}
with sampling period $\Delta T$. (This discretization is classical.)



## For left-invariant outputs

Consider the pre-observer
\begin{align*}
\widetilde g_k^- &= \widetilde g_{k-1}\exp(\Delta T\zeta_{k-1}), \\
\widetilde g_k &= \widetilde g_k^-\exp{\big(L(y_k,\widetilde y_k^-)\big)}\\
\widetilde y_k^- &= \phi_{\widetilde g_k}(\gamma),
\end{align*}
where $L:M\times M\to G$ is the innovation term. Define $u_{k-1}\triangleq \exp(\Delta T\zeta_{k-1})$, the a priori error $e_k^-\triangleq (\widetilde g_k^-)^{-1}g_k$, and the a posteriori error $e_k\triangleq \widetilde g_k^{-1}g_k$. Then
\begin{align*}
e_k^- &= (\widetilde g_k^-)^{-1} g_k = u_{k-1}^{-1}\widetilde g_{k-1}^{-1} g_{k-1}u_{k-1}
= u_{k-1}^{-1}e_{k-1}u_{k-1},\\
e_k &= \widetilde g_k^{-1} g_k = \exp{\big(-L(y_k,\widetilde y_k^-)\big)}e_k^-= \exp{\big(-L(y_k,\widetilde y_k^-)\big)}I_{u_{k-1}^{-1}}e_{k-1}.
\end{align*}

If $L$ is $G$-invariant, i.e., $L\big(\phi_g(y_1),\phi_g(y_2)\big)=L(y_1,y_2)$ for all $g\in G$, then using $y_k=\phi_{g_k}(\gamma)$ and $\widetilde y_k=\phi_{\widetilde g_k}(\gamma)$ one obtains $L(y_k,\widetilde y_k)=L\big(\phi_{e_k}(\gamma),\gamma\big)$, so the error dynamics are autonomous (time invariant).



## For right-invariant outputs

Consider the pre-observer
\begin{align*}
\widetilde g_k^- &= \widetilde g_{k-1}\exp(\Delta T\zeta_{k-1}),\\
\widetilde g_k &= \exp{\big(L(y_k,\widetilde y_k^-)\big)}\widetilde g_k^-, \\
\widetilde y_k^- &= \phi_{\widetilde g_k}(\gamma).
\end{align*}
With $u_{k-1}\triangleq \exp(\Delta T\zeta_{k-1})$ and the right-invariant errors $e_k^-\triangleq g_k(\widetilde g_k^-)^{-1}$ and $e_k\triangleq g_k\widetilde g_k^{-1}$, the error dynamics satisfy
\begin{align*}
e_k^- &= g_k(\widetilde g_k^-)^{-1} = g_{k-1}u_{k-1}u_{k-1}^{-1}\widetilde g_{k-1}^{-1} = e_{k-1},\\
e_k &= g_k\widetilde g_k^{-1} = e_k^-\exp{\big(-L(y_k,\widetilde y_k^-)\big)}= e_{k-1}\exp{\big(-L(y_k,\widetilde y_k^-)\big)}.
\end{align*}
If $L$ is $G$-invariant, one similarly gets $L(y_k,\widetilde y_k)=L\big(\phi_{e_k}(\gamma),\gamma\big)$, hence the error dynamics are autonomous.



# Intrinsic Extended Kalman Filter on Lie Groups

## Continuous-Time Intrinsic EKF on Lie Groups

Let $G$ be an $n$-dimensional Lie group with Lie algebra $G$. Let $(g,\zeta)\in G\times G$ and let $\phi:G\times M\to M$ be a left-invariant linear action on an $m$-dimensional vector space $M$ (so, for each $g\in G$, $\phi_g\in \mathrm{GL}(M)$ is linear). We recall the identities
\begin{align*}
g\exp(\zeta)g^{-1} &= \exp{\big(\mathrm{Ad}g\cdot \zeta\big)}, \\
\phi_{\exp(\zeta)} &= \exp{\big(T_e\phi \circ \zeta\big)}.
\end{align*}
In particular, if $M=G$ and $\phi=\mathrm{Ad}$ then $\mathrm{Ad}\,{\exp(\zeta)}=\exp(\mathrm{ad}\zeta)$.

We consider left-invariant Markov processes $g(t)\in G$ and $y(t)\in M$ given by
\begin{align*}
\dot g &= g\cdot\big(\zeta + n_\zeta\big),\\
y &= \phi_{g^{-1}}(\gamma) + n,
\end{align*}
where $\zeta(t)$ and $\gamma\in M$ are known, while $n_\zeta(t)$ and $n(t)$ are zero-mean Gaussian white noises with covariances $\Sigma_g$ and $\Sigma_y$, respectively.

Define
\begin{align*}
A(t) &\triangleq -\,\mathrm{ad}{\zeta(t)},\qquad\\
H(t) &\triangleq -\,\phi_{\widetilde g^{-1}}\!\big( (T_e\phi \circ \mathrm{Ad}{\widetilde g})\,\eta_e(\gamma)\big),
\end{align*}
and consider the intrinsic EKF
\begin{align*}
\dot{\widetilde{g}} &= \widetilde{g}\cdot\big(\zeta + K(t)(y-\widetilde{y})\big), \\
\widetilde y &= \phi_{\widetilde{g}^{-1}}(\gamma),
\end{align*}
with Riccati and gain
\begin{align*}
\dot P &= A P + P A^\top - P H^\top \Sigma_y^{-1} H P + \Sigma_\zeta,\qquad\\
K &= P H^\top \Sigma_y^{-1}.
\end{align*}
If $(A(t),H(t))$ is uniformly observable, the above filter ensures convergence in the sense stated below.

\begin{align*}
\dot{e}_g
&= e_g \cdot \Big( (I - \mathrm{Ad}_{e_g^{-1}})\zeta
- \mathrm{Ad}_{e_g^{-1}}K(t)y_e
+ n_\zeta \Big) \\
y_e
&= \phi_{\tilde{g}^{-1}}\Big( \big(\phi_{\tilde{g}e_g^{-1}\tilde{g}^{-1}} - I\big)(\gamma) \Big)
+ n
\end{align*}


Introduce the log-error $\eta_e$ and the corresponding output error $y_e$. One obtains the exact error dynamics
\begin{align*}
\dot\eta_e &= -\mathrm{ad}\zeta\eta_e
-\Big(\tfrac{\mathrm{ad}{\eta_e}}{\exp(\mathrm{ad}{\eta_e})-I}\Big)K(t)y_e
+\Big(\tfrac{\exp(\mathrm{ad}{\eta_e})\mathrm{ad}{\eta_e}}{\exp(\mathrm{ad}{\eta_e})-I}\Big)n_\zeta, \\
y_e &= \phi_{\widetilde g^{-1}}\Big(\big(\exp(-T_e\phi \circ \mathrm{Ad}{\widetilde g}\cdot \eta_e)-I\big)\gamma\Big) + n.
\end{align*}
Linearizing for small $|\eta_e|$ gives
\begin{align*}
\dot\eta_e &= \big(A(t)-K(t)H(t)\big)\eta_e + n_\zeta - K(t)n
+\Big(\tfrac{1}{2}\mathrm{ad}{\eta_e}+O(|\eta_e|^2)\Big)n_\zeta + O(|\eta_e|^2)n, \\
y_e &= H(t)\eta_e + n + O(|\eta_e|^2),
\end{align*}
so that the mean $m_{\eta_e}(t)=\mathbb{E}[\eta_e(t)]$ and covariance $P(t)=\mathbb{E}[\eta_e\eta_e^\top]$ evolve as
\begin{align*}
\dot m_{\eta_e} &= \big(A(t)-K(t)H(t)\big)m_{\eta_e} + \mathbb{E}[O(|\eta_e|^2)], \\
\dot P &= \big(A(t)-K(t)H(t)\big)P + P\big(A(t)-K(t)H(t)\big)^\top
P H^\top \Sigma_y^{-1} H P + \Sigma_\zeta \\
&\quad + \tfrac{1}{4}\mathbb{E}\big(\mathrm{ad}{\eta_e}\Sigma\zeta\mathrm{ad}_{\eta_e}^\top\big)
\mathbb{E}[O(|\eta_e|^3)].
\end{align*}

Under uniform observability and with $K(t)$ the Kalman gain for $(A(t),H(t),\Sigma_p,\Sigma)$, we have $m_{\eta_e}(t)\to 0$ and, writing $\varepsilon\triangleq \eta_e-m_{\eta_e}$ (mean zero, small covariance), the  Kalman filter property ensures that
that the covariance of $\varepsilon(t)$ converges to a constant and is small. Thus since $e_g = e^{\eta_e} = e^{(m_{\eta_e}+\varepsilon)} \approx e^{m_{\eta_e}}e^{\varepsilon}$ we have that
$\lim_{t\to\infty}e^{g(t)} = e^{\varepsilon}$ and hence that
group error locally converges to
\begin{align*}
\lim_{t\to\infty}\widetilde g = g(t)e^{-\varepsilon(t)}.
\end{align*}


## Discretized EKF on Lie Groups

We consider a Lie group $G$ with Lie algebra $\mathfrak{g}$ and a left-invariant linear action $\phi:G\times M\to M$ on a vector space $M$ (so each $\phi_g\in \mathrm{GL}(M)$).

Consider the discretized noisy rigid body kinematics
\begin{align*}
g_k &= g_{k-1}\exp{\big(\Delta T\zeta_{k-1}\big)},\\
y_k &= \phi_{g_k^{-1}}(\gamma)+z_k,\qquad p(z_k)=\mathscr{N}(0,\Sigma_m)
\end{align*}
This is a left invariant system on $G$ with a right invariant output.

Define the filter model
\begin{align*}
\widetilde g_k^- &= \widetilde g_{k-1}\exp{\left(\Delta T\left(\zeta_{k-1}+w_{k-1}\right)\right)},\qquad p(w_{k-1})=\mathscr{N}(0,\Sigma_p)\\
\widetilde g_k &= \exp{\big(L(y_k,\widetilde y_k^-)\big)}\widetilde g_k^-,\\
\widetilde y_k^- &= \phi_{{\widetilde{g}_k^-}^{-1}}(\gamma),
\end{align*}
where $\widetilde g_k^-$ should be interpreted as the predicted
random variable generated by the stochastic filter model (in direct
analogy with the Euclidean prediction), and
$L:M\times M\to \mathscr{G}$ is the innovation term. Define $u_{k-1}\triangleq \exp(\Delta T\zeta_{k-1})$, the a priori error $e_k^-\triangleq g_k(\widetilde g_k^-)^{-1}$, and the a posteriori error $e_k\triangleq g_k\widetilde g_k^{-1}$. Then
\begin{align*}
e_k^- &= g_k(\widetilde{g}_k^-)^{-1} ,\\
e_k &= g_k\widetilde{g}_k^{-1} =
g_k{\widetilde{g}_k^-}^{-1}\exp{\big(-L(y_k,\widetilde y_k^-)\big)}\\
&=
g_{k-1}\exp{\big(\Delta T\zeta_{k-1}\big)}
\exp{\left(-\Delta T\left(\zeta_{k-1}+w_{k-1}\right)\right)}
\widetilde{g}_{k-1}^{-1}\exp{\big(-L(y_k,\widetilde y_k^-)\big)}\\
&=
g_{k-1}\widetilde{g}_{k-1}^{-1}\mathbb{I}_{\widetilde{g}_{k-1}}
\left(\exp{\big(\Delta T\zeta_{k-1}\big)}
\exp{\left(-\Delta T\left(\zeta_{k-1}+w_{k-1}\right)\right)}\right)\exp{\big(-L(y_k,\widetilde y_k^-)\big)}\\
&=e_{k-1}\mathbb{I}_{\widetilde{g}_{k-1}}
\Big(\exp{\left(\Delta T\zeta_{k-1}\right)}
\exp{\left(-\Delta T\left(\zeta_{k-1}+w_{k-1}\right)\right)}\Big)\exp{\big(-L(y_k,\widetilde y_k^-)\big)}
\end{align*}


For the right-invariant output map, using the invariance property of $L$, we obtain
$$
\begin{aligned}
L(y_k,\widetilde y_k^-)
&=
L\left(
\phi_{g_k^{-1}}(\gamma),
\phi_{(\widetilde g_k^-)^{-1}}(\gamma)
\right)\\
&=
L\left(
\phi_{g_k}\phi_{g_k^{-1}}(\gamma),
\phi_{g_k}\phi_{(\widetilde g_k^-)^{-1}}(\gamma)
\right)\\
&=
L\left(
\gamma,
\phi_{g_k(\widetilde g_k^-)^{-1}}(\gamma)
\right)\\
&=
L\left(
\gamma,
\phi_{e_k^-}(\gamma)
\right).
\end{aligned}
$$


---

Define $e_k=e^{\eta_k}$ and also recall
\begin{align*}
\log(\exp X\,\exp Y)=X+\Phi(X)\,Y+O(\|Y\|^2)
\end{align*}
with
\begin{align*}
\displaystyle \Phi(\eta)=\frac{\mathrm{ad}\eta\,e^{\mathrm{ad}\eta}}{e^{\mathrm{ad}\eta}-I}
= I+\sum_{m=1}^{\infty}\frac{(-1)^{m+1}}{m(m+1)}\big(e^{\mathrm{ad}_\eta}-I\big)^m.
\end{align*}

Note that
\begin{align*}\Big(\exp{\left(\Delta T\zeta_{k-1}\right)}
\exp{\left(-\Delta T\left(\zeta_{k-1}+w_{k-1}\right)\right)}\Big)&=
\exp{\Big(
\Delta T\zeta_{k-1} - \Delta T\Phi(\Delta T\zeta_{k-1})\left(\zeta_{k-1}+w_{k-1}\right)+O(||\Delta T\zeta_{k-1}+\sqrt{\Delta T}w_{k-1}||^2)   
\Big)}.
\end{align*}
Let
\begin{align*}
Y&\triangleq
\Big(
\Delta T\zeta_{k-1} - \Delta T\Phi(\Delta T\zeta_{k-1})\left(\zeta_{k-1}+w_{k-1}\right)+O(||\Delta T\zeta_{k-1}+\sqrt{\Delta T}w_{k-1}||^2)    
\Big)
\end{align*}

---

Then we have
\begin{align*}
\exp{\eta_k} &=  
\exp{(\eta_{k-1})}\mathbb{I}_{\widetilde{g}_{k-1}}\exp{(Y)}\exp{\big(-L(y_k,\widetilde y_k^-)\big)},\\
&=  
\exp{(\eta_{k-1})}\exp{\left(\operatorname{Ad}_{\widetilde{g}_{k-1}}Y\right)}\exp{\big(-L(y_k,\widetilde y_k^-)\big)},\\
&=  
\exp{(\eta_{k-1})}\exp{\left(
\operatorname{Ad}_{\widetilde{g}_{k-1}}Y-\Phi\left(\operatorname{Ad}_{\widetilde{g}_{k-1}}Y\right)L(y_k,\widetilde y_k^-)
+O\left(||L(y_k,\widetilde y_k^-)||^2\right)
\right)},\\
&=\exp{\left(\eta_{k-1}-\Phi(\eta_{k-1})
\left(
\operatorname{Ad}_{\widetilde{g}_{k-1}}Y-\Phi\left(\operatorname{Ad}_{\widetilde{g}_{k-1}}Y\right)L(y_k,\widetilde y_k^-)
+O\left(||L(y_k,\widetilde y_k^-)||^2\right)
\right)+H.O.T.
\right)}
\end{align*}

Recalling
\begin{align*}
\Phi(\eta) = I + \frac{1}{2} \operatorname{ad}_\eta + \frac{1}{12} \operatorname{ad}_\eta^2 + O(\|\eta\|^3)
\end{align*}
and noting that $L(y_k,\widetilde y_k^-)\sim \eta_k$
we have retaining the $\eta_{k-1}$ order terms only
\begin{align*}
\eta_k &=  
\eta_{k-1}+\Phi(\eta_{k-1})
\left(
\operatorname{Ad}_{\widetilde{g}_{k-1}}Y-\Phi\left(\operatorname{Ad}_{\widetilde{g}_{k-1}}Y\right)L(y_k,\widetilde y_k^-)
+O\left(||L(y_k,\widetilde y_k^-)||^2\right)
\right)+H.O.T.\\
&=  
\eta_{k-1}+\left(I + \frac{1}{2} \operatorname{ad}_{\eta_{k-1}}\right)
\left(
\operatorname{Ad}_{\widetilde{g}_{k-1}}Y-
\left(I + \frac{1}{2} \operatorname{ad}_{\operatorname{Ad}_{\widetilde{g}_{k-1}}Y}\right)L(y_k,\widetilde y_k^-)
\right)+H.O.T.
\end{align*}

Rearranging and grouping the $O(\|L\|^2)$, $O(\|w\|^2)$, and
$O(\|L\|\|w\|)$ terms, we have
\begin{align*}
\eta_k
&=
\eta_{k-1}
+ \Phi(\eta_{k-1})
\Big(
  -\Delta T\operatorname{Ad}_{\widetilde g_{k-1}}
  \Phi(-\Delta T\zeta_{k-1})\,w_{k-1}
\\
&\hspace{3.3cm}
  -
  \Phi\!\big(
  -\Delta T\operatorname{Ad}_{\widetilde g_{k-1}}
  \Phi(-\Delta T\zeta_{k-1})\,w_{k-1}
  \big)
  L(y_k,\widetilde y_k^-)
\Big)
\\
&\qquad
+ O(\|L\|^2,\|w\|^2,\|L\|\|w\|).
\end{align*}

---

Neglecting higher-order cross and quadratic terms
$O(\|L\|\|w\|)$, $O(\|L\|^2)$, and $O(\|w\|^2)$,
while keeping the finite-step operator $\Phi(-\Delta T\zeta_{k-1})$ exact,
the error recursion reduces to
\begin{align*}
\eta_k
&=
\eta_{k-1}
+ \Phi(\eta_{k-1})
\Big(
  -\Delta T\operatorname{Ad}_{\widetilde g_{k-1}}
  \Phi(-\Delta T\zeta_{k-1})\,w_{k-1}
\Big)
- \Phi(\eta_{k-1})\,L(y_k,\widetilde y_k^-)
+ O(\|L\|^2,\|w\|^2,\|L\|\|w\|).
\end{align*}
Here, $\Phi(\eta_{k-1}) = I + \tfrac{1}{2}\operatorname{ad}_{\eta_{k-1}} + O(\|\eta_{k-1}\|^2)$
and $\Phi(-\Delta T\zeta_{k-1})$ is retained in closed form to preserve the exact
finite-step propagation of the process dynamics.

Now let us expand the output map. Since
$$
y_k=\phi_{g_k^{-1}}(\gamma),
\qquad
\widetilde y_k^-=\phi_{(\widetilde g_k^-)^{-1}}(\gamma),
$$
and since $L$ is invariant with respect to the action $\phi$, we obtain
\begin{align*}
L(y_k,\widetilde y_k^-)
&=
L\left(
\phi_{g_k^{-1}}(\gamma),
\phi_{(\widetilde g_k^-)^{-1}}(\gamma)
\right)\\
&=
L\left(
\phi_{g_k}\phi_{g_k^{-1}}(\gamma),
\phi_{g_k}\phi_{(\widetilde g_k^-)^{-1}}(\gamma)
\right)\\
&=
L\left(
\gamma,
\phi_{g_k(\widetilde g_k^-)^{-1}}(\gamma)
\right)\\
&=
L\left(
\gamma,
\phi_{e_k^-}(\gamma)
\right).
\end{align*}


\begin{align*}
e_k^- &= g_k(\widetilde{g}_k^-)^{-1} =g_{k-1}\exp{\left(\Delta T \zeta_{k-1}\right)}\exp{\left(-\Delta T\left( \zeta_{k-1}+w_{k-1}\right)\right)}\widetilde{g}_{k-1}^{-1}\\
&=e_{k-1}\mathbb{I}_{\widetilde{g}_{k-1}}\Big(\exp{\left(\Delta T \zeta_{k-1}\right)}\exp{\left(-\Delta T\left( \zeta_{k-1}+w_{k-1}\right)\right)}\Big)\\
&=e_{k-1}\mathbb{I}_{\widetilde{g}_{k-1}}
\Big(\exp{\left(\Delta T \zeta_{k-1}
-\Delta T \Phi\left(\Delta T \zeta_{k-1}\right)\left(\zeta_{k-1}+w_{k-1}\right)
+O\left(\Delta T^2||\zeta_{k-1}w_{k-1}||^2\right)\right)}\Big)
\end{align*}

Using the BCH expansion about $w_{k-1}=0$, we have
\begin{align*}
\exp(\Delta T\zeta_{k-1})
\exp\!\left(-\Delta T(\zeta_{k-1}+w_{k-1})\right)
&=
\exp\!\left(
-\Delta T\Phi(-\Delta T\zeta_{k-1})w_{k-1}
+
O(\|w_{k-1}\|^2)
\right).
\end{align*}
Therefore,
\begin{align*}
e_k^-
&=
e_{k-1}
\mathbb{I}_{\widetilde g_{k-1}}
\left[
\exp\!\left(
-\Delta T\Phi(-\Delta T\zeta_{k-1})w_{k-1}
+
O(\|w_{k-1}\|^2)
\right)
\right]\\
&=
e_{k-1}
\exp\!\left(
-\Delta T
\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1})w_{k-1}
+
O(\|w_{k-1}\|^2)
\right).
\end{align*}

Substituting $e_{k-1}=\exp(\eta_{k-1})$ and using the BCH expansion,
we obtain
$$
\eta_k^-
=
\eta_{k-1}
-
\Phi(\eta_{k-1})
\Delta T
\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1})w_{k-1}
+
O(\|w_{k-1}\|^2).
$$

Neglecting deterministic higher-order terms and higher-order noise terms gives
the linearized discrete error propagation
$$
\eta_k^-
=
\eta_{k-1}
-
\Delta T
\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1})w_{k-1}.
$$

Assume that, by design, the innovation has the linearization
$$
L(y_k,\widetilde y_k^-)
=
K_k\left(
H_k\left(\eta_{k-1}+G_{k-1}w_{k-1}\right)
+
z_k
\right),
$$
where
$$
G_{k-1}
\triangleq
-\Delta T\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1}).
$$
Here
$$
\Gamma_\phi
:=
\left.
\frac{d}{d\varepsilon}
\phi_{\exp(\varepsilon\xi)}(\gamma)
\right|_{\varepsilon=0},
\qquad
J_L
:=
\left.D_2L(\gamma,\cdot)\right|_{\cdot=\gamma},
\qquad
H_k:=J_L\Gamma_\phi .
$$

$$
\begin{aligned}
\eta_k
&=
\eta_{k-1}
+
G_{k-1}w_{k-1}
-
K_kH_k\eta_{k-1}
-
K_kH_kG_{k-1}w_{k-1}
-
K_kz_k\\
&=
(I-K_kH_k)\eta_{k-1}
+
(I-K_kH_k)G_{k-1}w_{k-1}
-
K_kz_k .
\end{aligned}
$$

**Note that these are in the exact form of the dicrete linear Kalman filter with $A_{k-1}=I$.**



### The DEKF Equations



Prediction (state on the group and covariance):
\begin{align*}
\widetilde g_k^- &= \widetilde g_{k-1}\exp{\left(\Delta T\zeta_{k-1}\right)}, \\
P_k^- &= A_{k-1} P_{k-1} A_{k-1}^{\top} + G_{k-1}\Sigma_pG_{k-1}^{\top}.
\end{align*}


Predicted output: consistent with the continuous-time relation $\widetilde y=\phi_{\widetilde g^{-1}}(\gamma)$, we use
\begin{align*}
\widetilde y_k^- = \phi_{{\widetilde{g}_k^-}^{-1}}(\gamma).
\end{align*}


Correction (group update and covariance update):
\begin{align*}
K_k &= P_k^- H_k^{\top}\big(H_k P_k^- H_k^{\top} + \Sigma_m\big)^{-1}, \\
P_k &= \big(I - K_k H_k\big)P_k^-, \\
\widetilde g_k &= \exp{\big(L(y_k,\widetilde y_k^-)\big)}\widetilde g_k^-,\\
\end{align*}
Here the innovation $L(y_k,\widetilde y_k^-)$ is assumed to have the linearization $L \approx K_k H_k\eta_{k}^-$
Note that these are in the linearized form of the discrete Kalman filter with $A_{k-1}=I$.

## Application to Rigid Body Motion Estimation

### Rigidbody Kinematics

Rigid-body kinematics on $SE(3)$ are given by
\begin{align*}
\dot g = g \cdot \zeta, \tag{54}
\end{align*}
where $g\in SE(3)=SO(3)\rtimes\mathbb{R}^3$ and $\zeta\in\mathfrak{se}(3)$. Writing out the blocks,
\begin{align*}
g &=
\begin{bmatrix}
R & o\\
0 & 1
\end{bmatrix},\qquad
\zeta =
\begin{bmatrix}
\widehat{\Omega} & V\\
0 & 0
\end{bmatrix},
\end{align*}
with $R\in SO(3)$, $o,V\in\mathbb{R}^3$, and $\widehat{\Omega}\in\mathfrak{so}(3)$. The dot “$\cdot$” denotes ordinary matrix multiplication; $g$ represents the change of frame from a fixed frame $e$ to a body-fixed frame $b$.  

For $\zeta=(\Omega,V)$ and $\xi=(\Phi,U)$ in $\mathfrak{se}(3)$, the adjoint action is
\begin{align*}
\operatorname{ad}_{\zeta} \xi = (\Omega\times\Phi, \Omega\times U - \Phi\times V),
\end{align*}
so that, in block form,
\begin{align*}
\operatorname{ad}\zeta =
\begin{bmatrix}
\widehat{\Omega} & 0\\
\widehat{V} & \widehat{\Omega}
\end{bmatrix}.
\end{align*}

The left action $\phi:SE(3)\times\mathbb{R}^4\to\mathbb{R}^4$ is simple homogeneous multiplication:
\begin{align*}
\phi_g(x) &=
\begin{bmatrix}
R & o\\
0 & 1
\end{bmatrix}
\begin{bmatrix}
x\\
\alpha
\end{bmatrix},
\quad
\text{with } g=(R,o)\in SE(3),\; x\in\mathbb{R}^3,\; \alpha\in\mathbb{R}.
\end{align*}

Points $\gamma=[x\:\: 1]^\top$ represent a fixed physical point's homogeneous coordinates; the orbit $\phi_g(\gamma)$ collects its different perspectives. For $\zeta=(\Omega,V)$,
\begin{align*}
(T_e\phi\circ \zeta)(\gamma) &=
\begin{bmatrix}
\widehat{\Omega} & V\\
0 & 0
\end{bmatrix}
\begin{bmatrix}
x\\
1
\end{bmatrix}.
\end{align*}

A key term that appears in the intrinsic linearization is
\begin{align*}
\phi_{\widetilde g^{-1}}\big( (T_e\phi\circ \operatorname{Ad}_{\widetilde g})\,\zeta \big)(\gamma)
&=
\zeta\widetilde g^{-1}\gamma\\
&=
\begin{bmatrix}
-\widetilde R^\top \widehat{(x-\widetilde o)}\widetilde R & I
\end{bmatrix}
\begin{bmatrix}
\Omega\\
V
\end{bmatrix},
\end{align*}
with $\widetilde g=(\widetilde R,\widetilde o)$. This leads directly to the explicit structure used in $H_k$ for the SLAM example below.

### Attitude Estimation from IMUs

#### Without Gyro Bias

In this case take $G=SO(3)$, $M=\mathfrak{g}\simeq\mathbb{R}^3$, and let the action be the adjoint $\phi=\mathrm{Ad}$ so that $\mathrm{Ad}_R x = Rx$ for $R\in SO(3)$, $x\in\mathbb{S}^2\subset \mathbb{R}^3$.

The Lie bracket is $ \mathrm{ad}_\Omega \Phi = \widehat{\Omega}\Phi = \Omega\times \Phi$. The measurement stacks two known inertial directions $e_1,e_2\in\mathbb{S}^2\subset \mathbb{R}^3$:
\begin{align*}
y_k =
\begin{bmatrix}
R_k^{T} e_1\\
R_k^{T} e_2
\end{bmatrix}.
\end{align*}
Using $\phi_{R^{T}}\circ (T_e\phi\circ \mathrm{Ad}_R)(\Omega)$, one finds the key identity (acting on any $x\in\mathbb{R}^3$):
\begin{align*}
\big(\phi_{R^{T}}\circ T_e\phi\circ \mathrm{Ad}_R\cdot \Omega\big)x
= -(R^{T}\widehat{x}\,R)\Omega,
\end{align*}
which yields the linearization used in $H_k$.

Discrete intrinsic EKF on $SO(3)$:
\begin{align*}
\widetilde R_k^{-} &= \widetilde R_{k-1}\exp{\big(\Delta T\left(\widehat{\Omega_{k-1}-{{\Omega}_{b}}_{k-1}}\right)\big)}, \\
\widetilde R_k &= \widetilde R_k^{-}\exp{\widehat{\Big(K_k\big(y_k-\widetilde y_k^-\big)\Big)}},
\end{align*}
with covariance prediction/gain update
\begin{align*}
P_k^{-} &= A_{k-1} P_{k-1} A_{k-1}^{T} + G_{k-1}\Sigma_qG_{k-1}^{T}, \\
K_k &= P_k^{-} H_k^{T}\big(H_k P_k^{-} H_k^{T} + \Sigma_m\big)^{-1}, \\
P_k &= (I - K_k H_k)\,P_k^-. \qquad
\end{align*}
and the discrete linearizations
\begin{align*}
A_{k-1} = I_{3\times 3},\qquad
G_{k-1} = -\Delta T\,\widetilde{R}_{k-1}\Phi(-\Delta T\, {\Omega}_{k-1}),\qquad
H_k =
\begin{bmatrix}
\widehat{{\widetilde{R}_k^-}^{\top}e_1}\\
\widehat{{\widetilde{R}_k^-}^{\top}e_2}
\end{bmatrix}.
\end{align*}
The predicted output stacks the two directions in the estimated/body frame:
\begin{align*}
\widetilde y_k^- =
\begin{bmatrix}
{\widetilde{R}_k^-}^{\top} e_1\\
{\widetilde{R}_k^-}^{\top} e_2
\end{bmatrix}.
\end{align*}
These equations are the attitude-only instance of the intrinsic discrete EKF on Lie groups specialized to $SO(3)$ with two vector measurements.


Since
$$
\Phi(-\Delta T\Omega_{k-1})=I_{3\times 3}+O(\Delta T),
$$
we may use the first-order approximation
$$
G_{k-1}
\approx
-\Delta T\,\widetilde R_{k-1}.
$$

##### Simulation Example

In [ ]:
import numpy as np


def generate_noisy_so3_direction_data(
    T=500,
    dt=0.01,
    omega_body=np.array([0.4, 0.2, 0.1]),
    inertial_directions=None,
    sigma_gyro=0.01,
    sigma_y=0.03,
    R0=None,
    seed=7,
):
    """
    Generate synthetic SO(3) attitude data with noisy gyro inputs and
    noisy vector-direction measurements.

    Measurement model:
        y_k = [R_k^T e_1; R_k^T e_2; ...] + noise

    Returns
    -------
    R_true : (T,3,3)
        True attitude trajectory.
    Omega_data : (T,3)
        Noisy gyro measurements.
    Y_data : (T,3m)
        Noisy stacked direction measurements.
    Y_clean : (T,3m)
        Noise-free stacked direction measurements.
    """

    rng = np.random.default_rng(seed)

    if inertial_directions is None:
        inertial_directions = np.array([
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
        ])

    E = np.asarray(inertial_directions, dtype=float)
    m = E.shape[0]

    if R0 is None:
        R = np.eye(3)
    else:
        R = np.asarray(R0, dtype=float).reshape(3, 3)

    omega_body = np.asarray(omega_body, dtype=float).reshape(3)

    def hat(x):
        x = np.asarray(x, dtype=float).reshape(3)
        return np.array([
            [0.0, -x[2], x[1]],
            [x[2], 0.0, -x[0]],
            [-x[1], x[0], 0.0],
        ])

    def exp_so3(omega):
        omega = np.asarray(omega, dtype=float).reshape(3)
        theta = np.linalg.norm(omega)
        W = hat(omega)

        if theta < 1e-12:
            return np.eye(3) + W + 0.5 * W @ W

        A = np.sin(theta) / theta
        B = (1.0 - np.cos(theta)) / theta**2
        return np.eye(3) + A * W + B * W @ W

    R_true = []
    Omega_data = []
    Y_clean = []
    Y_data = []

    for k in range(T):
        # True attitude propagation
        R = R @ exp_so3(dt * omega_body)

        # Noisy gyro measurement
        Omega_k = omega_body + rng.normal(0.0, sigma_gyro, size=3)

        # Clean stacked direction measurement
        y_clean_k = np.concatenate([R.T @ e for e in E])

        # Noisy measurement
        y_noisy_k = y_clean_k + rng.normal(0.0, sigma_y, size=3 * m)

        # Optional: renormalize each measured direction
        for i in range(m):
            block = y_noisy_k[3*i:3*(i+1)]
            norm = np.linalg.norm(block)
            if norm > 1e-12:
                y_noisy_k[3*i:3*(i+1)] = block / norm

        R_true.append(R.copy())
        Omega_data.append(Omega_k)
        Y_clean.append(y_clean_k)
        Y_data.append(y_noisy_k)

    return (
        np.asarray(R_true),
        np.asarray(Omega_data),
        np.asarray(Y_data),
        np.asarray(Y_clean),
    )

In [ ]:
E = np.array([
    #[1.0, 0.0, 0.0],
    #[0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])

R_true, Omega_data, Y_data, Y_clean = generate_noisy_so3_direction_data(
    T=500,
    dt=0.01,
    omega_body=np.array([0.4, 0.2, 0.1]),
    inertial_directions=E,
    sigma_gyro=0.01,
    sigma_y=0.03,
    seed=7,
)

In [ ]:
kf = sims.LinearKF(use_joseph=True, symmetrize=True)
iekf = sims.SO3IMUSensorFusionEKF(kf=kf, inertial_directions=E, dt=0.01)

results = iekf.run_filter(
    R0_plus=np.eye(3),
    P0=1e-2 * np.eye(3),
    Omegas=Omega_data,
    Ys=Y_data,
    Sigma_q=1e-4 * np.eye(3),
    Sigma_m=1e-2 * np.eye(3),
)

fig = iekf.plot_measurements(Y_data, results["yhat"])
fig.show()

#### With Gyro Bias

The augmented local error state is
$$
\eta_k
=
\begin{bmatrix}
\eta_{R,k}\\
\eta_{b,k}
\end{bmatrix}
\in \mathbb R^6,
$$
where $\eta_{R,k}\in\mathbb R^3$ is the attitude error and
$\eta_{b,k}\in\mathbb R^3$ is the gyro-bias error.

Discrete intrinsic EKF on $SO(3)$ with gyro bias:
\begin{align*}
\widetilde R_k^{-}
&=
\widetilde R_{k-1}
\exp\!\left(
\Delta T\,\left(\widehat{\Omega_{k-1}-\widetilde\Omega_{b,k-1}}\right)
\right),\\
\widetilde\Omega_{b,k}^{-}
&=
\widetilde\Omega_{b,k-1}.
\end{align*}


$$
\begin{bmatrix}
\delta_{R,k}\\
\delta_{b,k}
\end{bmatrix}
=
K_k(y_k-\widetilde y_k^-),
$$

\begin{align*}
\widetilde R_k
&=
\widetilde R_k^{-}\exp(\widehat{\delta_{R,k}}),\\
\widetilde\Omega_{b,k}
&=
\widetilde\Omega_{b,k}^{-}+\delta_{b,k}.
\end{align*}

\begin{align*}
P_k^{-}
&=
A_{k-1}P_{k-1}A_{k-1}^{T}
+
G_{k-1}\Sigma_qG_{k-1}^{T},\\
K_k
&=
P_k^{-}H_k^{T}
\big(H_kP_k^{-}H_k^{T}+\Sigma_m\big)^{-1},\\
P_k
&=
(I-K_kH_k)P_k^-.
\end{align*}

Here
$$
A_{k-1}
=
\begin{bmatrix}
I_{3\times3} & -\Delta T\,I_{3\times3}\\
0 & I_{3\times3}
\end{bmatrix}.
$$

$$
G_{k-1}
=
\begin{bmatrix}
-\Delta T\,\widetilde R_{k-1}
& 0\\
0 & I_{3\times3}
\end{bmatrix}.
$$

$$
H_k
=
\begin{bmatrix}
\widehat{(\widetilde R_k^-)^{T}e_1} & 0_{3\times3}\\
\widehat{(\widetilde R_k^-)^{T}e_2} & 0_{3\times3}
\end{bmatrix}.
$$

$$
\widetilde y_k^-
=
\begin{bmatrix}
(\widetilde R_k^-)^T e_1\\
(\widetilde R_k^-)^T e_2
\end{bmatrix}.
$$

In [ ]:
# hat/exp_so3 are re-defined here at notebook scope: in the example above
# (cell with generate_noisy_so3_direction_data) they are local to that
# function only, so they are not otherwise available globally.
def hat(x):
    x = np.asarray(x, dtype=float).reshape(3)
    return np.array([
        [0.0, -x[2], x[1]],
        [x[2], 0.0, -x[0]],
        [-x[1], x[0], 0.0],
    ])


def exp_so3(omega):
    omega = np.asarray(omega, dtype=float).reshape(3)
    theta = np.linalg.norm(omega)
    W = hat(omega)
    if theta < 1e-12:
        return np.eye(3) + W + 0.5 * W @ W
    A = np.sin(theta) / theta
    B = (1.0 - np.cos(theta)) / theta**2
    return np.eye(3) + A * W + B * W @ W


def geodesic_angle(R1, R2):
    """Angle (rad) of the rotation R1^T R2, i.e. the geodesic distance on SO(3)."""
    M = R1.T @ R2
    c = (np.trace(M) - 1.0) / 2.0
    c = np.clip(c, -1.0, 1.0)
    return np.arccos(c)


def generate_noisy_so3_direction_data_with_gyro_bias(
    T=2000,
    dt=0.01,
    omega_body=np.array([0.4, 0.2, 0.1]),
    inertial_directions=None,
    b_true=np.array([0.05, -0.03, 0.02]),
    sigma_gyro=0.01,
    sigma_y=0.03,
    R0=None,
    seed=7,
):
    """
    Same generative model as generate_noisy_so3_direction_data above, but the
    gyro measurement is additionally corrupted by a constant (unknown) bias
    b_true, as in the 'With Gyro Bias' model:

        Omega_k^m = omega_body + b_true + noise.

    Returns R_true, Omega_data, Y_data, Y_clean, b_true.
    """
    rng = np.random.default_rng(seed)

    if inertial_directions is None:
        inertial_directions = np.array([
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
        ])

    E = np.asarray(inertial_directions, dtype=float)
    m = E.shape[0]

    R = np.eye(3) if R0 is None else np.asarray(R0, dtype=float).reshape(3, 3)
    omega_body = np.asarray(omega_body, dtype=float).reshape(3)
    b_true = np.asarray(b_true, dtype=float).reshape(3)

    R_true, Omega_data, Y_clean, Y_data = [], [], [], []

    for k in range(T):
        # True attitude propagation (driven by the true, bias-free rate)
        R = R @ exp_so3(dt * omega_body)

        # Noisy, BIASED gyro measurement
        Omega_k = omega_body + b_true + rng.normal(0.0, sigma_gyro, size=3)

        # Clean / noisy stacked direction measurements
        y_clean_k = np.concatenate([R.T @ e for e in E])
        y_noisy_k = y_clean_k + rng.normal(0.0, sigma_y, size=3 * m)
        for i in range(m):
            block = y_noisy_k[3 * i:3 * (i + 1)]
            norm = np.linalg.norm(block)
            if norm > 1e-12:
                y_noisy_k[3 * i:3 * (i + 1)] = block / norm

        R_true.append(R.copy())
        Omega_data.append(Omega_k)
        Y_clean.append(y_clean_k)
        Y_data.append(y_noisy_k)

    return (
        np.asarray(R_true),
        np.asarray(Omega_data),
        np.asarray(Y_data),
        np.asarray(Y_clean),
        b_true,
    )


class SO3IMUSensorFusionGyroBiasEKF:
    """
    Discrete intrinsic EKF on SO(3) with gyro-bias augmentation, implementing
    the 'With Gyro Bias' equations derived above:

        R~_k^-      = R~_{k-1} exp(dt (Omega_{k-1} - b~_{k-1})),
        b~_k^-      = b~_{k-1},
        A_{k-1}     = [[I, -dt I], [0, I]]                (6x6),
        G_{k-1}     = [[-dt R~_{k-1}, 0], [0, I]]          (6x6),
        H_k         = stack_i [ hat((R~_k^-)^T e_i),  0 ]  (3m x 6),
        y~_k^-      = stack_i (R~_k^-)^T e_i,
        K_k         = P_k^- H_k^T (H_k P_k^- H_k^T + Sigma_m)^-1,
        [delta_R; delta_b] = K_k (y_k - y~_k^-),
        R~_k        = R~_k^- exp(hat(delta_R)),
        b~_k        = b~_k^- + delta_b,
        P_k         = (I - K_k H_k) P_k^-.
    """

    def __init__(self, inertial_directions, dt):
        self.E = np.asarray(inertial_directions, dtype=float)
        self.m = self.E.shape[0]
        self.dt = dt

    def run_filter(self, R0_plus, b0_plus, P0, Omegas, Ys, Sigma_q, Sigma_m):
        dt = self.dt
        E = self.E
        T = Omegas.shape[0]
        I3 = np.eye(3)

        R = np.asarray(R0_plus, dtype=float).reshape(3, 3).copy()
        b = np.asarray(b0_plus, dtype=float).reshape(3).copy()
        P = np.asarray(P0, dtype=float).copy()

        R_hist = np.zeros((T, 3, 3))
        b_hist = np.zeros((T, 3))
        yhat_hist = np.zeros((T, 3 * self.m))
        P_trace_hist = np.zeros(T)

        for k in range(T):
            Omega_k = Omegas[k]

            # ---- Prediction ----
            R_minus = R @ exp_so3(dt * (Omega_k - b))
            b_minus = b.copy()

            A = np.block([[I3, -dt * I3],
                          [np.zeros((3, 3)), I3]])
            G = np.block([[-dt * R, np.zeros((3, 3))],
                          [np.zeros((3, 3)), I3]])
            P_minus = A @ P @ A.T + G @ Sigma_q @ G.T

            # ---- Predicted output ----
            yhat_minus = np.concatenate([R_minus.T @ e for e in E])

            # ---- Linearized output map H_k ----
            H = np.vstack([
                np.hstack([hat(R_minus.T @ e), np.zeros((3, 3))])
                for e in E
            ])

            # ---- Correction ----
            S = H @ P_minus @ H.T + Sigma_m
            K = P_minus @ H.T @ np.linalg.inv(S)
            innovation = Ys[k] - yhat_minus
            delta = K @ innovation
            delta_R, delta_b = delta[:3], delta[3:]

            R = R_minus @ exp_so3(delta_R)
            b = b_minus + delta_b
            P = (np.eye(6) - K @ H) @ P_minus

            R_hist[k] = R
            b_hist[k] = b
            yhat_hist[k] = yhat_minus
            P_trace_hist[k] = np.trace(P)

        return dict(R=R_hist, b=b_hist, yhat=yhat_hist, P_trace=P_trace_hist)


In [ ]:
# Reuse the two inertial directions and rate from the bias-free example above,
# but now corrupt the gyro with a constant unknown bias b_true.
E_bias = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
])
b_true = np.array([0.05, -0.03, 0.02])

R_true_b, Omega_data_b, Y_data_b, Y_clean_b, b_true = generate_noisy_so3_direction_data_with_gyro_bias(
    T=3000,
    dt=0.01,
    omega_body=np.array([0.4, 0.2, 0.1]),
    inertial_directions=E_bias,
    b_true=b_true,
    sigma_gyro=0.01,
    sigma_y=0.03,
    seed=7,
)

iekf_bias = SO3IMUSensorFusionGyroBiasEKF(inertial_directions=E_bias, dt=0.01)

Sigma_gyro = 0.02**2 * np.eye(3)
Sigma_bias = 1e-5**2 * np.eye(3)
Sigma_q = np.block([
    [Sigma_gyro, np.zeros((3, 3))],
    [np.zeros((3, 3)), Sigma_bias],
])
Sigma_m = 0.03**2 * np.eye(3 * E_bias.shape[0])

results_bias = iekf_bias.run_filter(
    R0_plus=np.eye(3),
    b0_plus=np.zeros(3),
    P0=1e-2 * np.eye(6),
    Omegas=Omega_data_b,
    Ys=Y_data_b,
    Sigma_q=Sigma_q,
    Sigma_m=Sigma_m,
)

rotation_error = np.array([
    geodesic_angle(R_true_b[k], results_bias["R"][k]) for k in range(len(R_true_b))
])
bias_error = np.linalg.norm(results_bias["b"] - b_true[None, :], axis=1)

print(f"True gyro bias:       {b_true}")
print(f"Estimated gyro bias:  {results_bias['b'][-1]}")
print(f"Final rotation error (rad): {rotation_error[-1]:.5f}")
print(f"Final bias estimation error (norm): {bias_error[-1]:.5f}")


In [ ]:
time_axis = np.arange(len(rotation_error)) * 0.01

fig = go.Figure()
fig.add_trace(go.Scatter(x=time_axis, y=rotation_error, mode="lines", name="Rotation error (rad)"))
fig.update_layout(
    title="Gyro-Bias EKF: Attitude Estimation Error vs Time",
    xaxis_title="time (s)",
    yaxis_title="geodesic error (rad)",
)
fig.show()

fig2 = go.Figure()
labels = ["b_x", "b_y", "b_z"]
colors = ["red", "green", "blue"]
for i in range(3):
    fig2.add_trace(go.Scatter(
        x=time_axis, y=results_bias["b"][:, i], mode="lines",
        name=f"estimated {labels[i]}", line=dict(color=colors[i]),
    ))
    fig2.add_trace(go.Scatter(
        x=time_axis, y=[b_true[i]] * len(time_axis), mode="lines",
        name=f"true {labels[i]}", line=dict(color=colors[i], dash="dash"),
    ))
fig2.update_layout(
    title="Gyro-Bias EKF: Bias Estimate Convergence",
    xaxis_title="time (s)",
    yaxis_title="gyro bias (rad/s)",
)
fig2.show()


In [ ]:
Y_data

In [ ]:
Omega_data

##### Example with real data

[The Zurich Urban Micro Aerial Vehicle Dataset](https://www.kaggle.com/datasets/mrisdal/zurich-urban-micro-aerial-vehicle)

In [ ]:
!pip install "git+https://github.com/mugalan/data-analysis-tool.git"

In [ ]:
from data_analysis import DataInspector, PlottingMethods
inspector = DataInspector()

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mrisdal/zurich-urban-micro-aerial-vehicle")

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df_gyro=pd.read_csv(path+"/RawGyro.csv")
df_accel=pd.read_csv(path+"/RawAccel.csv")

In [ ]:
df_merged = pd.merge(df_accel, df_gyro, on='Timpstemp', how='inner', suffixes=('_accel', '_gyro'))
display(df_merged.head())

In [ ]:
inspector.df=df_merged

In [ ]:
column_names=[' x_accel', ' y_accel', ' z_accel', ' x_gyro', ' y_gyro', ' z_gyro']
inspector.plot_numerical(column_names=column_names)

In [ ]:
Omega_data=df_gyro[[' x', ' y', ' z']].to_numpy()
raw_acc=df_accel[[' x', ' y', ' z']].to_numpy()

In [ ]:
Y_data = raw_acc/ np.linalg.norm(raw_acc, axis=1, keepdims=True)

In [ ]:
Y_data

In [ ]:
def lowpass_accel(acc, alpha=0.02):
    acc_lp = np.zeros_like(acc)
    acc_lp[0] = acc[0]

    for k in range(1, len(acc)):
        acc_lp[k] = (1 - alpha) * acc_lp[k-1] + alpha * acc[k]

    return acc_lp

In [ ]:
low_pass_acc=lowpass_accel(raw_acc)

In [ ]:
Y_data = low_pass_acc/ np.linalg.norm(low_pass_acc, axis=1, keepdims=True)

In [ ]:
E = np.array([
    [0.0, 0.0, -1.0],   # gravity direction for your accelerometer convention
])

kf = sims.LinearKF(use_joseph=True, symmetrize=True)

iekf = sims.SO3IMUSensorFusionBiasEKF(
    kf=kf,
    inertial_directions=E,
    dt=0.01,
)

Sigma_gyro = 0.02**2 * np.eye(3)
Sigma_bias = 1e-5**2 * np.eye(3)

Sigma_q = np.block([
    [Sigma_gyro, np.zeros((3, 3))],
    [np.zeros((3, 3)), Sigma_bias],
])

Sigma_m = 0.18**2 * np.eye(3)

results = iekf.run_filter(
    R0_plus=np.eye(3),
    b0_plus=np.zeros(3),
    P0=np.diag([1e-2, 1e-2, 1e-2, 1e-4, 1e-4, 1e-4]),
    Omegas=Omega_data,
    Ys=Y_data,
    Sigma_q=Sigma_q,
    Sigma_m=Sigma_m,
)

fig = iekf.plot_measurements(Y_data, results["yhat"])
fig.show()

fig_b = iekf.plot_bias(results)
fig_b.show()

### SLAM

Let $e$ be a fixed world frame and $p_i$ for $i=1,\dots,N$ be fixed landmarks. Write $\gamma_i=[x_i\:\:1]^\top\in\mathbb{R}^4$ for their homogeneous coordinates in $e$, and let $X_i$ denote the coordinates in the body frame related by $g\in SE(3)$. The SLAM model is
\begin{align*}
\dot g = g\cdot(\zeta + n_p), \qquad
y_i = \phi_{g^{-1}}(\gamma_i) + n_m,
\end{align*}
with $y_i=[X_i;1]^\top$, $n_p\sim\mathscr{N}(0,\Sigma_p)$, $n_m\sim\mathscr{N}(0,\Sigma_m)$. (Velocity biases can be added if needed.)

From the intrinsic linearization on $SE(3)$ one obtains the discrete EKF matrices (for sample time $\Delta T$):
\begin{align*}
A_{k-1}
&=I_{6\times 6},
\\
H_k
&=
\begin{bmatrix}
\widetilde R_k^\top \,\widehat{(x_1-\widetilde o_k)}\,\widetilde R_k & -I_{3\times 3}\\
\widetilde R_k^\top \,\widehat{(x_2-\widetilde o_k)}\,\widetilde R_k & -I_{3\times 3}\\
\vdots & \vdots\\
\widetilde R_k^\top \,\widehat{(x_m-\widetilde o_k)}\,\widetilde R_k & -I_{3\times 3}
\end{bmatrix},
\end{align*}
and
$$
G_{k-1}
=
-\Delta T
\operatorname{Ad}_{\widetilde g_{k-1}}
\Phi(-\Delta T\zeta_{k-1}),
$$
where, for $SE(3)$,
$$
\operatorname{Ad}_{\widetilde g_{k-1}}
=
\begin{bmatrix}
\widetilde R_{k-1} & 0\\
\widehat{\widetilde o}_{k-1}\widetilde R_{k-1} & \widetilde R_{k-1}
\end{bmatrix},
\qquad
\operatorname{ad}_{\zeta_{k-1}}
=
\begin{bmatrix}
\widehat{\Omega}_{k-1} & 0\\
\widehat V_{k-1} & \widehat{\Omega}_{k-1}
\end{bmatrix}.
$$
Thus, to first order in $\Delta T$,
$$
G_{k-1}
\approx
-\Delta T
\begin{bmatrix}
\widetilde R_{k-1} & 0\\
\widehat{\widetilde o}_{k-1}\widetilde R_{k-1} & \widetilde R_{k-1}
\end{bmatrix}.
$$


These plug into the intrinsic discrete EKF update
\begin{align*}
\widetilde g_k^-&=\widetilde g_{k-1}\exp(\Delta T\,\zeta_{k-1}),\\
\widetilde g_k&=\widetilde g_k^- \exp{\big(K_k(y_k-\widetilde y_k)\big)},
\end{align*}
with the usual $K_k$, $P_k$ recursions defined earlier.

### The IMU+GNSS sensor fusion problem

#### Without Bias

A representative application is rigid-body orientation/pose estimation with IMU and GNSS. Gyroscopes measure body-frame angular velocity $\Omega$, accelerometers measure $A_m$ (specific force), and GNSS provides $o,\dot o$. A convenient reformulation introduces
\begin{align*}
o_s(t)&\triangleq o(t)+\tfrac{1}{2}gt^2 e_3,\qquad\\
v_s(t)&\triangleq \dot o_s(t)=\dot o(t)+g t\,e_3,\qquad\\
R\,A_m&=\ddot o_s(t)=\ddot o(t)+g e_3,
\end{align*}
giving the continuous-time model
\begin{align*}
\dot R &= R\,\widehat\Omega, \\
\dot v_s &= R\,A_m, \\
\dot o_s &= v_s, \\
y_o &= o_s, \\
y_v &= v_s.
\end{align*}
Defining the homogeneous state
\begin{align*}
X &\triangleq\;
\begin{bmatrix}
R & v_s\\
0 & 1
\end{bmatrix},\qquad\\
\zeta &\triangleq
\begin{bmatrix}
\widehat\Omega & A_m\\
0 & 0
\end{bmatrix},\qquad\\
\gamma_v &\triangleq\;
\begin{bmatrix}
0_{3\times 1}\\
1
\end{bmatrix},
\end{align*}
the dynamics/output compactly read
\begin{align*}
\dot X &= X\,\zeta, \\
y_v &= X\,\gamma_v.
\end{align*}

\begin{align*}
\dot o_s &= v_s= X\,\gamma_v, \\
y_o &= o_s.
\end{align*}

A first-order discretization of $ \dot X = X\zeta $ gives
\begin{align*}
X_{k+1} &= X_k \exp{\big(\Delta t\zeta_k\big)}.
\end{align*}

With the block structure of $\zeta_k$, the matrix exponential splits as
\begin{align*}
\exp{\big(\Delta t\zeta_k\big)}
&=
\begin{bmatrix}
\exp{\big(\Delta t\widehat\Omega_k\big)} & \Delta t A^m_k\\
0 & 1
\end{bmatrix}.
\end{align*}

Therefore, the discrete updates in these shifted variables are
\begin{align*}
R_{k+1} &= R_k\exp{\big(\Delta t\widehat\Omega_k\big)},\\
v_{s,k+1} &= v_{s,k} + \Delta tR_k\,A^m_k,\\
o_{s,k+1} &= o_{s,k} + \Delta tv_{s,k}.
\end{align*}

Returning to the original variables, with gravity acting along $e_3$, we obtain the discrete updates
$$
R_{k+1}
\approx
R_k\exp(\Delta t\,\widehat{\Omega}_k),
$$
$$
v_{k+1}
=
v_k
-
g\Delta t\,e_3
+
\Delta t\,R_kA_k^m,
$$
and, using the corresponding second-order position update,
$$
o_{k+1}
=
o_k
+
\Delta t\,v_k
-
\frac{1}{2}g(\Delta t)^2e_3
+
\frac{1}{2}(\Delta t)^2R_kA_k^m .
$$

Notes: $\widehat\Omega_k$ denotes the skew-symmetric matrix of the angular velocity sample $\Omega_k$, $A^m_k$ is the measured specific force at step $k$, $e_3=(0,0,1)^\top$.

---


The attitude, shifted velocity, and shifted position are denoted by
$$
R_k\in SO(3),\qquad
v_{s,k}\in\mathbb R^3,\qquad
o_{s,k}\in\mathbb R^3.
$$

The shifted variables are defined by
$$
o_{s,k}
\triangleq
o_k+\frac{1}{2}g t_k^2 e_3,
\qquad
v_{s,k}
\triangleq
v_k+g t_k e_3,
$$
where
$$
v_k=\dot o_k,
\qquad
e_3=(0,0,1)^\top .
$$

The Lie-group state containing attitude and shifted velocity is
$$
X_k
\triangleq
\begin{bmatrix}
R_k & v_{s,k}\\
0 & 1
\end{bmatrix}.
$$

The full navigation state is then written as
$$
\mathscr X_k
\triangleq
(X_k,o_{s,k}),
$$
or equivalently,
$$
\mathscr X_k
=
(R_k,v_{s,k},o_{s,k}).
$$

The corresponding local error state is
$$
\eta_k
\triangleq
\begin{bmatrix}
\eta_{R,k}\\
\eta_{v,k}\\
\eta_{o,k}
\end{bmatrix}
\in\mathbb R^9,
$$
where
$$
\eta_{R,k}\in\mathbb R^3
$$
is the attitude error in the Lie algebra, and
$$
\eta_{v,k},\eta_{o,k}\in\mathbb R^3
$$
are the shifted velocity and shifted position errors.

$$
A_{k-1}
=
\begin{bmatrix}
I_{3\times 3} & 0 & 0\\
0 & I_{3\times 3} & 0\\
0 & \Delta t\, I_{3\times 3} & I_{3\times 3}
\end{bmatrix}.
$$

$$
H_k
=
\begin{bmatrix}
0 & 0 & I_{3\times 3}\\
0 & I_{3\times 3} & 0
\end{bmatrix}.
$$

$$
G_{k-1}^{X}
=
-\Delta t\,
\operatorname{Ad}_{\widetilde X_{k-1}}
\Phi(-\Delta t\,\zeta_{k-1}),
$$


$$
\operatorname{Ad}_{\widetilde X_{k-1}}
=
\begin{bmatrix}
\widetilde R_{k-1} & 0\\
\widehat{\widetilde v}_{s,k-1}\widetilde R_{k-1} & \widetilde R_{k-1}
\end{bmatrix}.
$$



$$
G_{k-1}
=
\begin{bmatrix}
G_{k-1}^{X}\\
0_{3\times 6}
\end{bmatrix}
=
\begin{bmatrix}
-\Delta t\,
\operatorname{Ad}_{\widetilde X_{k-1}}
\Phi(-\Delta t\,\zeta_{k-1})\\
0_{3\times 6}
\end{bmatrix}.
$$

---

For
$$
\zeta_{k-1}
=
\begin{bmatrix}
\Omega_{k-1}\\
A^m_{k-1}
\end{bmatrix},
$$
we have
$$
\operatorname{ad}_{\zeta_{k-1}}
=
\begin{bmatrix}
\widehat{\Omega}_{k-1} & 0\\
\widehat{A^m}_{k-1} & \widehat{\Omega}_{k-1}
\end{bmatrix}.
$$

Therefore,
$$
\Phi(-\Delta t\,\zeta_{k-1})
=
I_{6\times 6}
-
\frac{\Delta t}{2}\operatorname{ad}_{\zeta_{k-1}}
+
\frac{(\Delta t)^2}{12}\operatorname{ad}_{\zeta_{k-1}}^2
+
O((\Delta t)^3).
$$

Equivalently,
$$
\Phi(-\Delta t\,\zeta_{k-1})
=
I_{6\times 6}
-
\frac{\Delta t}{2}
\begin{bmatrix}
\widehat{\Omega}_{k-1} & 0\\
\widehat{A^m}_{k-1} & \widehat{\Omega}_{k-1}
\end{bmatrix}
+
\frac{(\Delta t)^2}{12}
\begin{bmatrix}
\widehat{\Omega}_{k-1} & 0\\
\widehat{A^m}_{k-1} & \widehat{\Omega}_{k-1}
\end{bmatrix}^2
+
O((\Delta t)^3).
$$

#### With Bias

We now consider a more complete IMU--GNSS sensor fusion model including
gyroscope and accelerometer biases. Let
$$
R_k \in SO(3), \qquad v_k \in \mathbb R^3, \qquad o_k \in \mathbb R^3
$$
denote the attitude, velocity, and position of the rigid body expressed with
respect to an inertial frame. Let $e_3=(0,0,1)^\top$, and let $g>0$
denote the gravitational acceleration magnitude.

The IMU provides gyroscope and accelerometer measurements
$$
\Omega_k^m = \Omega_k + b_{\Omega,k} + n_{\Omega,k},
$$
$$
A_k^m = A_k + b_{A,k} + n_{A,k},
$$
where
$$
b_{\Omega,k}\in \mathbb R^3,
\qquad
b_{A,k}\in \mathbb R^3
$$
are the gyroscope and accelerometer biases, and
$$
n_{\Omega,k}\sim \mathscr N(0,\Sigma_{\Omega}),
\qquad
n_{A,k}\sim \mathscr N(0,\Sigma_A)
$$
are zero-mean measurement noises.

Therefore, the bias-corrected angular velocity and specific force are
$$
\Omega_k = \Omega_k^m - b_{\Omega,k} - n_{\Omega,k},
$$
$$
A_k = A_k^m - b_{A,k} - n_{A,k}.
$$

The continuous-time nominal inertial navigation model is
$$
\dot R = R\widehat{\Omega},
$$
$$
\dot v = RA - ge_3,
$$
$$
\dot o = v.
$$

Including the measured IMU signals and biases, this becomes
$$
\dot R
=
R\left(\widehat{\Omega^m-b_{\Omega}-n_{\Omega}}\right),
$$
$$
\dot v
=
R(A^m-b_A-n_A)-ge_3,
$$
$$
\dot o=v.
$$

The bias dynamics are commonly modeled as random walks:
$$
\dot b_{\Omega}=w_{\Omega},
\qquad
\dot b_A=w_A,
$$
where
$$
w_{\Omega}\sim \mathscr N(0,\Sigma_{b_\Omega}),
\qquad
w_A\sim \mathscr N(0,\Sigma_{b_A}).
$$

Using a sampling period $\Delta t$, the corresponding discrete-time
propagation model is
$$
R_{k+1}
=
R_k
\exp\!\left(
\Delta t\,
\widehat{\Omega_k^m-b_{\Omega,k}}
\right),
$$
$$
v_{k+1}
=
v_k
+
\Delta t\,R_k(A_k^m-b_{A,k})
-
g\Delta t\,e_3,
$$
$$
o_{k+1}
=
o_k
+
\Delta t\,v_k
+
\frac{1}{2}(\Delta t)^2 R_k(A_k^m-b_{A,k})
-
\frac{1}{2}g(\Delta t)^2e_3.
$$

The bias states evolve as
$$
b_{\Omega,k+1}
=
b_{\Omega,k}
+
w_{\Omega,k},
$$
$$
b_{A,k+1}
=
b_{A,k}
+
w_{A,k}.
$$

Equivalently, the full navigation state may be written as
$$
X_k
=
\left(
R_k,\,
v_k,\,
o_k,\,
b_{\Omega,k},\,
b_{A,k}
\right).
$$

The GNSS measurements typically provide position and, in some systems,
velocity. Thus a standard GNSS measurement model is
$$
y_k^{\mathrm{GNSS}}
=
\begin{bmatrix}
o_k\\
v_k
\end{bmatrix}
+
z_k,
$$
where
$$
z_k\sim \mathscr N(0,\Sigma_{\mathrm{GNSS}}).
$$

If GNSS provides only position, the measurement model reduces to
$$
y_k^{\mathrm{GNSS}}
=
o_k+z_k,
\qquad
z_k\sim \mathscr N(0,\Sigma_o).
$$

Thus the complete IMU--GNSS model consists of the propagation equations
$$
\boxed{
R_{k+1}
=
R_k
\exp\!\left(
\Delta t\,
\widehat{\Omega_k^m-b_{\Omega,k}}
\right)
}
$$
$$
\boxed{
v_{k+1}
=
v_k
+
\Delta t\,R_k(A_k^m-b_{A,k})
-
g\Delta t\,e_3
}
$$
$$
\boxed{
o_{k+1}
=
o_k
+
\Delta t\,v_k
+
\frac{1}{2}(\Delta t)^2R_k(A_k^m-b_{A,k})
-
\frac{1}{2}g(\Delta t)^2e_3
}
$$
$$
\boxed{
b_{\Omega,k+1}=b_{\Omega,k}+w_{\Omega,k},
\qquad
b_{A,k+1}=b_{A,k}+w_{A,k}
}
$$
together with the GNSS measurement equation
$$
\boxed{
y_k^{\mathrm{GNSS}}
=
\begin{bmatrix}
o_k\\
v_k
\end{bmatrix}
+
z_k
}
$$
or, for position-only GNSS,
$$
\boxed{
y_k^{\mathrm{GNSS}}
=
o_k+z_k.
}
$$

# References

[1] K. C. Wolfe, M. Mashner, and G. S. Chirikjian, “Bayesian fusion on Lie groups,” *Journal of Algebraic Statistics*, 2(1):75–97, 2011. [Link](https://rpk.lcsr.jhu.edu/publications/)

[2] S. Bonnable, P. Martin, and E. Salan, “Invariant extended Kalman filter: theory and application to a velocity-aided attitude estimation problem,” *Proc. 48th IEEE CDC/28th CCC*, pp. 1297–1304, Dec 2009. [Link](https://doi.org/10.1109/CDC.2009.5399990)

[3] S. Bonnabel, “Left-invariant extended Kalman filter and attitude estimation,” *Proc. 46th IEEE CDC*, pp. 1027–1032, Dec 2007. [Link](https://doi.org/10.1109/CDC.2007.4434662)

[4] G. Bourmaud, R. Mégret, A. Giremus, and Y. Berthoumieu, “Discrete extended Kalman filter on Lie groups,” *EUSIPCO 2013*, pp. 1–5, Sept 2013. [Link](https://hal.science/hal-00903252/document)

[5] G. S. Chirikjian, “Information theory on Lie groups and mobile robotics applications,” *Proc. 2010 IEEE ICRA*, pp. 2751–2757, May 2010. [Link](https://rpk.lcsr.jhu.edu/publications/)

[6] Y. Wang and G. S. Chirikjian, “Error propagation on the Euclidean group with applications to manipulator kinematics,” *IEEE Trans. Robotics*, 22(4):591–602, Aug 2006. [Link](https://ieeexplore.ieee.org/document/1673946)

[7] M. J. Piggott and V. Solo, “Stochastic numerical analysis for Brownian motion on SO(3),” *Proc. 53rd IEEE CDC*, pp. 3420–3425, Dec 2014. [Link](https://doi.org/10.1109/CDC.2014.7039919)

[8] O. Tuzel, F. Porikli, and P. Meer, “Learning on Lie groups for invariant detection and tracking,” *Proc. 2008 IEEE CVPR*, pp. 1–8, June 2008. [Link](https://doi.org/10.1109/CVPR.2008.4587521)

[9] A. Barrau and S. Bonnabel, “Intrinsic filtering on Lie groups with applications to attitude estimation,” *IEEE Trans. Automatic Control*, 60(2):436–449, Feb 2015. [Link](https://doi.org/10.1109/TAC.2014.2342911)

[10] S. Bonnabel and A. Barrau, “An intrinsic Cramér–Rao bound on SO(3) for (dynamic) attitude filtering,” *Proc. 54th IEEE CDC*, pp. 2158–2163, Dec 2015. [Link](https://dblp.org/rec/conf/cdc/BonnabelB15)

[11] A. Barrau and S. Bonnabel, “The invariant extended Kalman filter as a stable observer,” *IEEE Trans. Automatic Control*, 62(4):1797–1812, Apr 2017. [Link](https://doi.org/10.1109/TAC.2016.2594085)

[12] C. Lageman, J. Trumpf, and R. Mahony, “Gradient-like observers for invariant dynamics on a Lie group,” *IEEE Trans. Automatic Control*, 55(2):367–377, Feb 2010. [Link](https://doi.org/10.1109/TAC.2009.2034937)

[13] S. Bonnabel, P. Martin, and P. Rouchon, “Non-linear symmetry-preserving observers on Lie groups,” *IEEE Trans. Automatic Control*, 54(7):1709–1713, July 2009. [Link](https://doi.org/10.1109/TAC.2009.2020646)

[14] M. Izadi and A. K. Sanyal, “Rigid body attitude estimation based on the Lagrange–d’Alembert principle,” *Automatica*, 50(10):2570–2577, 2014. [Link](https://doi.org/10.1016/j.automatica.2014.08.010)

[15] S. Bonnabel and J. J. Slotine, “A contraction theory-based analysis of the stability of the deterministic extended Kalman filter,” *IEEE Trans. Automatic Control*, 60(2):565–569, Feb 2015. [Link](https://doi.org/10.1109/TAC.2014.2336991)
